# 固定OOF参考策略的完整成本与SOXX对比
补齐既有2020–2022确定性风险参考账户的全部成本，不重训、不改变参考策略。新建5个原生账户，复用2bp参考。明确不是当前完整生产策略或新的独立验证。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_oof_account_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,p['reference_source'],['inputs/plan','inputs/frame','analysis/native_reference']);action,aa=_source(root,p['action_source'],['inputs/plan','controller/source_recovery_selection'])
for key in ['tariff','initial_cash','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert source['inputs/plan'][key]==p[key]
frame=source['inputs/frame'];assert len(frame)==756 and frame.trade_date.dt.year.min()==2020 and frame.trade_date.dt.year.max()==2022
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);controller={'frozen_open_action_value_model':None,'frozen_recovery_model':None};selection=action['controller/source_recovery_selection']
books={('reference',2):source['analysis/native_reference']};assert not books[('reference',2)]['authority'].recovery_action.any()
state=json.loads((card/'run_state.json').read_text()) if (card/'run_state.json').exists() else {}
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if state:kwargs.update(recorder_id=state['recorder_id'],resume=True)
with R.start(**kwargs):
 rec=R.get_recorder()
 if state:assert rec.load_object('inputs/plan')==p, 'resume plan differs'
 rec.save_objects(artifact_path='inputs',plan=p,source_audit=audit,action_audit=aa,frame=frame);rec.save_objects(artifact_path='analysis/native',reference_2=books[('reference',2)]);rec.log_params(new_fits=0,maximum_new_native_cases=5)
count=0
for family,cost in [('reference',0),('reference',5),('soxx',0),('soxx',2),('soxx',5)]:
 if f'{family}_{cost}' in state.get('completed_keys',[]):
  books[(family,cost)]=rec.load_object(f'analysis/native/{family}_{cost}');count+=1;continue
 replay=replay_full_path(frame,models=None,plan=p,slippage_bps=cost,controller=controller if family=='reference' else None,recovery_spec=recovery,action_spec=spec,selection=selection,benchmark=family=='soxx')
 if family=='reference':assert replay['replay'].execute_open_target.all() and not replay['replay'].recovery_action.any()
 book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=cost,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);books[(family,cost)]=book;rec.save_objects(artifact_path='analysis/native',**{f'{family}_{cost}':book});count+=1;rec.log_metrics(completed_new_native_cases=count)
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];bench=books[('soxx',cost)]['authority'];assert path.trade_date.equals(bench.trade_date);relative=np.log1p(path.strategy_return)-np.log1p(bench.strategy_return)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,bench),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean(),'mean_cash_pct':100*path.ending_cash_weight.mean(),'orders':len(book['orders'])})
 # Attribution follows the decision-year fold, not preceding signal year.
 annual.append(pd.DataFrame([{'year':int(year),'year_basis':'trade_year_fold_attribution',**paired_metrics(path.loc[path.trade_date.dt.year.eq(year)],bench.loc[bench.trade_date.dt.year.eq(year)])} for year in sorted(path.trade_date.dt.year.unique())]).assign(family=family,slippage_bps=cost))
 if family=='reference':uncertainty.append({'slippage_bps':cost,**stationary_bootstrap_excess(path,bench,mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);candidate=table[table.family.eq('reference')]
gates={'net_soxx_excess_all_costs':bool(candidate.relative_wealth.gt(1).all()),'drawdown_30pct':bool(candidate.max_drawdown_pct.ge(-30).all() and candidate.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(candidate.cvar5_pct.ge(candidate.benchmark_cvar5_pct).all()),'rolling_70pct':bool(candidate.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq('reference')].groupby('slippage_bps').relative_wealth.apply(lambda x:x.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'independent_validation':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/native/*']};summary={'experiment':card.name,'decision':'diagnostic_reference_not_promoted','gates':gates,'new_fits':0,'new_native_cases':count,'reused_native_cases':1,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'first_trade':str(frame.trade_date.min().date()),'last_exit':str(frame.exit_date.max().date()),'annual_attribution':p['annual_attribution'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=uncertainty,summary=summary);rec.log_metrics(registered_artifact_count=14)
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty)]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/p['reference_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# OOF参考账户补齐：2020–2022未通过净超额与风险门

本轮回到策略验证，0新拟合、5套新增Qlib原生账户，复用已有2bp参考。固定此前9个年度小配置模型的时间外预测、连续目标平滑及确定性风险规则，补齐0/5bp参考与0/2/5bp SOXX。不重新选择模型、参数、样本窗或风险阈值。本金10万美元，IBKR Pro Tiered统一费用投影，现金利息仍为0。

评价交易日2020-01-02至2022-12-30，756个连续持有区间，最后退出2023-01-03。年度归因按交易年，最后跨年收益归属其2022年起始持有区间；不是把2023年完整收益混入开发期。各成本账户持仓、现金、峰值和风险状态独立连续，无跨年账户重置。

| 滑点 | 参考净收益 | SOXX净收益 | 参考监控最大回撤 | 126日正超额比例 |
|---|---:|---:|---:|---:|
| 0bp | -5.06% | +42.06% | 32.88% | 28.37% |
| 2bp | -4.71% | +42.04% | 32.98% | 28.21% |
| 5bp | -6.08% | +42.01% | 33.12% | 26.94% |

2bp按交易年：2020参考-2.29%、SOXX+53.99%；2021参考+22.21%、SOXX+42.31%；2022参考-20.20%、SOXX-35.18%。仅2022相对少跌，不足以抵消前两年差距；三个成本的净超额、30%回撤、年度/滚动稳定及置信门均不通过。

这是既有动作标签生成参考的完整费用比较，参考不含学习型开盘/恢复控制器，以避免把后来训练的控制器放回早期年份。它不是完整生产策略，不能据此单独证明完整策略失败；缺少恢复模块也可能影响反弹参与。模型训练具有年度时间外资格，但结构已经经过历史研究选择，且源价格/费率仍是统一研究投影，不是未查看的独立数据或真实券商账单。

5个新原生账户已通过现金、股数、费用与净值对账。首次年度报告代码把trade_date写入signal_date列，触发因果日期门禁；修正为保持原列、按交易年分组，全部已存账户复用，未重复回测。恢复执行还核对原Recorder中的计划与当前计划一致。

决策diagnostic_reference_not_promoted。固定参考评估完成，不调参数挽救，也不把更简单的参考替代完整目标。下一信息需求是能在早期时点使用的完整控制器证据及配置在上涨/反弹阶段的错误来源；任何进一步训练必须时间隔离，不能据本表挑年份或把2023后参数回填2020。现金利息缺口保留为独立needs_data支线，暂停扩展该支线的来源工作，不将其视为本轮约47个百分点收益差的已证解释。

生产未改，稳定净SOXX超额目标未完成。完整报告、年度表及区块置信结果已登记Qlib Recorder，NAS冷恢复逐对象哈希通过。
